---
authors:
  - name: Tom Siegl
---

# MLP Foundations


## Towards MLPs

Today we will start by trying to solve the XOR classification problem using the linear modeling tools that we have learned about in previous sessions.
We will see what is and isn't possible with linear methods and that the multilayer perceptron (MLP) is a natural abstraction of combining linear models.

But first, have a look at the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

std = 0.01
X = np.concatenate([
    np.random.multivariate_normal([1, 1], np.eye(2)*std, 100),
    np.random.multivariate_normal([1, 0], np.eye(2)*std, 100),
    np.random.multivariate_normal([0, 0], np.eye(2)*std, 100),
    np.random.multivariate_normal([0, 1], np.eye(2)*std, 100),
])
y = np.concatenate([
    np.full(100, 1),
    np.full(100, 0),
    np.full(100, 1),
    np.full(100, 0)
])

In [ ]:
def plot_XOR_data(xs, ys, ax):
    ax.scatter(xs[y.astype(bool), 0], xs[y.astype(bool), 1], color="blue", label="Class 1")
    ax.scatter(xs[np.invert(y.astype(bool)), 0], xs[np.invert(y.astype(bool)), 1], color="red",  label="Class 0")
    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.legend()


def get_grid_points(grid_min, grid_max, grid_size):
    grid_vals = np.linspace(grid_min, grid_max, grid_size)
    xx, yy = np.meshgrid(grid_vals, grid_vals)
    return np.column_stack([xx.ravel(), yy.ravel()]), xx, yy


def plot_predictions_grid_regression(fig, ax, xx, yy, grid_min, grid_max, preds, levels=[0.5]):
    preds = np.reshape(preds, (len(xx), len(yy)))

    im = ax.imshow(preds[::-1, :], extent=(grid_min, grid_max, grid_min, grid_max))
    contour = ax.contour(xx, yy, preds, levels=levels, colors="black")
    ax.clabel(contour)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("predictions")


def plot_predictions_grid_classification(fig, ax, xx, yy, preds):
    preds = np.reshape(preds, (len(xx), len(yy)))
    
    cf = ax.contourf(xx, yy, preds, alpha=0.4, cmap=plt.cm.RdBu)
    
    cbar = fig.colorbar(cf, ax=ax)
    cbar.set_ticks([0, 1])
    cbar.set_label("predictions")


fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111)
plot_XOR_data(X, y, ax)
ax.set_title("XOR data")
plt.show()

We have seen the XOR problem in the lecture, where the famous result was stated, that this simple problem cannot be solved by a linear model alone.

Instead of using just one linear model, let's try to solve it by chaining linear models.
For starters, we have the following two linear models, that each correctly classify one blob of class 1 and class 0.

$$\hat{y}_1 = 1 + (-1) \cdot x_1 + (-1) \cdot x_2 = 1 - x_1 - x_2$$

$$\hat{y}_2 = -1 + 1 \cdot x_1 + 1 \cdot x_2 = -1 + x_1 + x_2$$

In [ ]:
from sklearn.linear_model import LinearRegression


def get_linear_models_class_1():
    lr1 = LinearRegression()
    lr1.intercept_ = 1
    lr1.coef_ = np.array([-1, -1])
    
    lr2 = LinearRegression()
    lr2.intercept_ = -1
    lr2.coef_ = np.array([1, 1])

    return lr1, lr2


def get_linear_models_binary():
    lr1 = LinearRegression()
    lr1.intercept_ = 0
    lr1.coef_ = np.array([0, 1])
    
    lr2 = LinearRegression()
    lr2.intercept_ = 0
    lr2.coef_ = np.array([1, 0])

    return lr1, lr2

In [ ]:
lr1, lr2 = get_linear_models_class_1()
# lr1, lr2 = get_linear_models_binary()  # in case you want to try something else

# print model parameters
print("Parameters of the first model:", f"beta^T = [{lr1.intercept_:.3f}, {lr1.coef_[0]:.3f}, {lr1.coef_[1]:.3f}]")
print("Parameters of the second model:", f"beta^T = [{lr2.intercept_:.3f}, {lr2.coef_[0]:.3f}, {lr2.coef_[1]:.3f}]")

In [ ]:
# get plot predictions
plot_min = -0.5
plot_max = 1.5
grid_points, xx, yy = get_grid_points(plot_min, plot_max, 100)

lr1_preds_grid = lr1.predict(grid_points)
lr2_preds_grid = lr2.predict(grid_points)


# plot
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

plot_predictions_grid_regression(fig, ax1, xx, yy, plot_min, plot_max, lr1_preds_grid)
plot_XOR_data(X, y, ax1)
ax1.set_title("Linear model 1")

plot_predictions_grid_regression(fig, ax2, xx, yy, plot_min, plot_max, lr2_preds_grid)
plot_XOR_data(X, y, ax2)
ax2.set_title("Linear model 2")

plt.tight_layout()
plt.show()

These two linear models both solve a subproblem of the overall problem, and it would be awesome if we could just combine them to form the overall solution.

The idea is simple: When predictions from model 1 are large, then it's probably class 1 and when predictions from model 2 are large, then it's probably class 2.
So let's try to train another linear model on the outputs of model 1 and 2 to solve the XOR problem.

In [ ]:
lr1_preds = lr1.predict(X)
lr2_preds = lr2.predict(X)
X_preds = np.stack([lr1_preds, lr2_preds], axis=-1)  # save predictions from models 1 and 2 for use as input in model 3

lr3 = LinearRegression().fit(X_preds, y)  # fit model 3 on predictions from models 1 and 2

# Print model parameters
print("Parameters of the second layer:", f"beta^T = [{lr3.intercept_:.3f}, {lr3.coef_[0]:.3f}, {lr3.coef_[1]:.3f}]")

In [ ]:
# plot
lr3_preds_grid = lr3.predict(np.stack([lr1_preds_grid, lr2_preds_grid], axis=-1))

fig = plt.figure(figsize=(6, 5))
ax1 = fig.add_subplot(111)

plot_predictions_grid_regression(fig, ax1, xx, yy, plot_min, plot_max, lr3_preds_grid)
plot_XOR_data(X, y, ax1)
ax1.set_title("Combined linear model")

plt.tight_layout()
plt.show()

Hm, this doesn't look right at all.

To find out why this fails, we will have to look at some equations.
Firstly we formalize our setup as:

$$\mathbf{a}^{(0)} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}, \qquad b^{(1)}_i = \hat{\beta}_{0, i}, \qquad W^{(1)}_i = \begin{bmatrix} \hat{\beta}_{1, i} & \hat{\beta}_{2, i} \end{bmatrix}, \qquad i \in \{1, 2\}$$

$$a^{(1)}_1 = W^{(1)}_1 \mathbf{a}^{(0)} + b^{(1)}_1, \qquad a^{(1)}_2 = W^{(1)}_2 \mathbf{a}^{(0)} + b^{(1)}_2$$

$$W^{(1)} = \begin{bmatrix} W^{(1)}_1 \\ W^{(1)}_2 \end{bmatrix}, \qquad \mathbf{b}^{(1)} = \begin{bmatrix} b^{(1)}_1 \\ b^{(1)}_2 \end{bmatrix}$$

$$\mathbf{a}^{(1)} = W^{(1)} \mathbf{a}^{(0)} + \mathbf{b}^{(1)} = \begin{bmatrix} a^{(1)}_1 \\ a^{(1)}_2 \end{bmatrix}$$

$$\mathbf{a}^{(2)} &= W^{(2)} \mathbf{a}^{(1)} + \mathbf{b}^{(2)}$$

:::{important} Tasks
1. Name the variables in the equations for the outputs of linear model 1, linear model 2 and the combined linear model.
2. Substitute and rewrite the last equation to obtain $\mathbf{a}^{(2)} = W' \mathbf{a}^{(0)} + b'$.
:::

:::{tip} Your answer
:class:dropdown
1. 
2. 
:::

As we can see, stacking linear models doesn't really work out as we wished.
The result is again just a linear function!
(But you already knew that from the lecture.)

Let's go back to our two linear models from the start.
They are supposed to classify the points according to the XOR function, so maybe we should treat the classification step as a part of them.

The following plots reflect this point of view, by showing the final predicted classes from each of these models.

In [ ]:
def threshold(x,  t=0.5):
    return x > t

lr1_preds_grid_classified = threshold(lr1.predict(grid_points))
lr2_preds_grid_classified = threshold(lr2.predict(grid_points))

In [ ]:
# plot
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

plot_predictions_grid_classification(fig, ax1, xx, yy, lr1_preds_grid_classified)
plot_XOR_data(X, y, ax1)
ax1.set_title("Linear Classifier 1")

plot_predictions_grid_classification(fig, ax2, xx, yy, lr2_preds_grid_classified)
plot_XOR_data(X, y, ax2)
ax2.set_title("Linear Classifier 2")

plt.tight_layout()
plt.show()

These class outputs are not the result of a purely linear function anymore, as they include the thresholding step.
So the argument from before, that chaining another model after these gives us still just a linear function overall, does not hold anymore.

Let's try again to combine these predictions using the same kind of model.

In [ ]:
lr1_preds_classified = threshold(lr1.predict(X))
lr2_preds_classified = threshold(lr2.predict(X))
X_preds_classified = np.stack([lr1_preds_classified, lr2_preds_classified], axis=-1)

lr3_classified = LinearRegression().fit(X_preds_classified, y)

# Print model parameters
print("Parameters of the second layer:", f"beta^T = [{lr3_classified.intercept_:.3f}, {lr3_classified.coef_[0]:.3f}, {lr3_classified.coef_[1]:.3f}]")

In [ ]:
# plot
lr3_preds_grid_classified = threshold(lr3_classified.predict(np.stack([lr1_preds_grid_classified, lr2_preds_grid_classified], axis=-1)))

fig = plt.figure(figsize=(6, 5))
ax1 = fig.add_subplot(111)

plot_predictions_grid_classification(fig, ax1, xx, yy, lr3_preds_grid_classified)
plot_XOR_data(X, y, ax1)
ax1.set_title("Combined linear classifier")

plt.tight_layout()
plt.show()

It worked!

Just one problem: We didn't train the whole thing.
During training, we would have to somehow make sure, that the first two linear models provide sensible features for the third model to use.

We know from the lecture, that MLPs are more complex than linear models and that they can be trained as a whole.
So let's have a look at how to interpret our current setup as an MLP.

:::{important} Tasks
1. How many hidden layers does our combined linear classifier have?
2. How many neurons does the first hidden layer have in our combined linear classifier?
3. What is the activation function in our combined linear classifier?
:::

:::{tip} Your answer
:class:dropdown
1. 
2. 
3. 
:::

We can see that our combined linear classifier model actually matches the structure of an MLP, so we have just constructed our first MLP model.

The lecture said that to train MLPs, usually gradient descent is used.
Let's have a look at what that means and how this affects our model.

:::{important} Tasks
1. What is the meaning of the direction and magnitude of the gradient $\nabla_{\theta} L$? (**Hint**: First think about what $L$ and $\theta$ are.)
2. How is the gradient used to improve the model in gradient descent?
3. Why is sigmoid a better choice than the threshold function as an activation function, when we want to train with gradient descent? (**Hint**: Look at the plots from the next code cell.)
:::

:::{tip} Your answer
:class:dropdown
1. 
2. 
3. 
:::

In [ ]:
def plot_activation(f, f_label, title, start, end, samples, ax):
    samples_1d = np.linspace(start, end, samples)
    
    ax.plot(samples_1d, f(samples_1d), label=f_label)
    ax.set_title(title)
    ax.set_xlabel(r"$W^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$")
    ax.set_ylabel(r"$a^{(l)}$")
    ax.legend()


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

plot_activation(threshold, r"$W^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)} > 0.5$", "Threshold function as activation function", 0, 1, 1000, ax1)
plot_activation(sigmoid, r"$\text{sigmoid}(W^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)})$", "Sigmoid function as activation function", -10, 10, 1000, ax2)

plt.tight_layout()
plt.plot()

The next cells use logistic regression models to implement the idea of replacing the threshold function with sigmoid in our combined model / MLP.

In [ ]:
from sklearn.linear_model import LogisticRegression


def get_logistic_models_class_1(xs, ys):
    logr1 = LogisticRegression().fit(xs, ys)
    logr1.intercept_ = 0.5
    logr1.coef_ = np.array([-1, -1])
    
    logr2 = LogisticRegression().fit(xs, ys)
    logr2.intercept_ = -1.5
    logr2.coef_ = np.array([1, 1])

    return logr1, logr2


def get_logistic_models_binary(xs, ys):
    logr1 = LogisticRegression().fit(xs, ys)
    logr1.intercept_ = -0.5
    logr1.coef_ = np.array([0, 1])
    
    logr2 = LogisticRegression().fit(xs, ys)
    logr2.intercept_ = -0.5
    logr2.coef_ = np.array([1, 0])

    return logr1, logr2

In [ ]:
logr1, logr2 = get_logistic_models_class_1(X, y)  # creates logistic regression models with the same parameters as linear models 1 and 2 before
#logr1, logr2 = get_logistic_models_binary(X, y)

logr1_preds_grid = logr1.predict_proba(grid_points)[:, 1]  # need to call predict_proba on sklearn's logistic regression models to get raw predictions (not yet thresholded)
logr2_preds_grid = logr2.predict_proba(grid_points)[:, 1]

In [ ]:
# plot
fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

plot_predictions_grid_regression(fig, ax1, xx, yy, plot_min, plot_max, logr1_preds_grid)
plot_XOR_data(X, y, ax1)
ax1.set_title("Logistic regression model 1")

plot_predictions_grid_regression(fig, ax2, xx, yy, plot_min, plot_max, logr2_preds_grid)
plot_XOR_data(X, y, ax2)
ax2.set_title("Logistic regression model 2")

plt.tight_layout()
plt.show()

In [ ]:
logr1_preds = logr1.predict_proba(X)[:, 1]
logr2_preds = logr2.predict_proba(X)[:, 1]

X_preds_log = np.stack([logr1_preds, logr2_preds], axis=-1)

logr3 = LogisticRegression().fit(X_preds_log, y)
print("Parameters of the second layer:", f"beta^T = [{logr3.intercept_[0]:.3f}, {logr3.coef_[0, 0]:.3f}, {logr3.coef_[0, 1]:.3f}]")

# for zero in-sample error
# logr3.intercept_ = np.array([-1.55])
# logr3.coef_ = np.array([[2, 2]])

In [ ]:
# plot
logr3_preds_grid = logr3.predict_proba(np.stack([logr1_preds_grid, logr2_preds_grid], axis=-1))[:, 1]

fig = plt.figure(figsize=(6, 5))
ax1 = fig.add_subplot(111)

plot_predictions_grid_regression(fig, ax1, xx, yy, plot_min, plot_max, logr3_preds_grid)
plot_XOR_data(X, y, ax1)
ax1.set_title("Combined logistic regression model")

plt.tight_layout()
plt.show()

We can observe that the result is really similar to the combined linear classifier, but the predictions are now smooth.
We still have to apply the threshold function at the end to obtain the final classes, but now we have sigmoid inside the model as a smooth alternative for the thresholding.
